# CISPO vs PPO through a tiny network, by hand

```
X (one prompt, 4 rows = 4 sampled responses) -> Z1 = X @ W1 -> H = tanh(Z1) -> logits = H @ W2 -> p = softmax(logits)
rollout (old weights): each row wrote token y, logp_old = log p_old[y]; reward -> GRPO advantage A (one per row)
training step (current weights): logp_y = log p[y],  r = exp(logp_y - logp_old)

REINFORCE: loss = mean(-A * logp_y)
PPO:       loss = mean(-min(r * A, clip(r, 0.8, 1.28) * A))
CISPO:     loss = mean(-w * A * logp_y),  w = clip(r, 0, 4) frozen (stop-gradient)
```

All three give the same gradient on the logits, `w * A * (p - y_onehot) / N`. They differ only in
`w`, one number per row: REINFORCE `w = 1`; PPO `w = r`, or `0` when the clip fires; CISPO
`w = clip(r, 0, 4)`. Everything after the logits (`dW2`, `dW1`) is the same code for all three.
Each response here is one token long, so `/ N` (N = 4 tokens) is also CISPO's token-level
normalization `1 / Σ|o_i|`.

In [1]:
import numpy as np
import torch
np.set_printoptions(precision=3, suppress=True)

TOKENS = ["A", "B", "C"]
X = np.array([[0.0, 1.0]] * 4)          # one prompt, 4 rows: 4 sampled responses
PPO_CLIP = (0.8, 1.28)                  # 1 - eps_low, 1 + eps_high (the eps_high our runs use)
CISPO_CLIP = (0.0, 4.0)                 # miles/Tinker CISPO defaults: no lower bound, cap 4


class Network:
    def __init__(self, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 1, (2, 4))   # input -> hidden. seed 0: [[ 0.126, -0.132, 0.640, 0.105],
                                             #                           [-0.536,  0.362, 1.304, 0.947]]
        self.W2 = rng.normal(0, 1, (4, 3))   # hidden -> scores. seed 0: [[-0.704, -1.265, -0.623],
                                             #                            [ 0.041, -2.325, -0.219],
                                             #                            [-1.246, -0.732, -0.544],
                                             #                            [-0.316,  0.412,  1.043]]

    def probs(self, X):
        z = np.tanh(X @ self.W1) @ self.W2
        e = np.exp(z - z.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def forward(self, X, y, logp_old, A, kind):
        # 1. Z1     = X @ W1
        # 2. H      = tanh(Z1)
        # 3. logits = H @ W2
        # 4. p      = softmax(logits)
        # 5. logp_y = log p[row, y]               log-prob of the token each row wrote
        # 6. r      = exp(logp_y - logp_old)      current vs rollout probability
        # 7. w      = per-row weight (REINFORCE / PPO / CISPO), a frozen number
        # 8. loss
        # Values in the comments: step 0 (current weights = rollout weights, seed 0).
        # All 4 rows share the prompt, so layers 1-4 are the same in every row.
        acts = {}
        b = X.shape[0]                                             # 4
        acts["X"], acts["y"], acts["A"], acts["b"] = X, y, A, b
        acts["Z1"] = X @ self.W1                                   # 1: each row [-0.536, 0.362, 1.304, 0.947]  (= row 2 of W1)
        acts["H"] = np.tanh(acts["Z1"])                            # 2: each row [-0.490, 0.347, 0.863, 0.738]
        acts["logits"] = acts["H"] @ self.W2                       # 3: each row [-0.950, -0.514, 0.530]
        e = np.exp(acts["logits"] - acts["logits"].max(axis=1, keepdims=True))
        acts["p"] = e / e.sum(axis=1, keepdims=True)               # 4: each row [0.144, 0.223, 0.633]
        logp_y = np.log(acts["p"][np.arange(b), y])                # 5: y = [A, B, C, C] -> [-1.937, -1.501, -0.457, -0.457]
        r = np.exp(logp_y - logp_old)                              # 6: step 0: [1, 1, 1, 1]  (same weights as the rollout)
                                                                   #    step 1: [1.456, 0.851, 0.949, 0.949]
        if kind == "reinforce":
            w = np.ones(b)
        elif kind == "ppo":
            # The clipped term wins the min only on the side A pushes toward. Then the loss
            # term is a constant and the row's gradient is 0. Otherwise d(-r*A) = -A * r * d(logp_y).
            lo, hi = PPO_CLIP
            clipped = ((A > 0) & (r > hi)) | ((A < 0) & (r < lo))
            w = np.where(clipped, 0.0, r)                          # 7: step 0: [1, 1, 1, 1]
                                                                   #    step 1: [0, 0.851, 0.949, 0.949]  row A: 1.456 > 1.28 with A > 0
        elif kind == "cispo":
            w = np.clip(r, *CISPO_CLIP)                            # 7: step 0: [1, 1, 1, 1]
                                                                   #    step 1: [1.456, 0.851, 0.949, 0.949]  nothing reaches the cap of 4
        acts["r"], acts["w"] = r, w
        self.acts = acts
        if kind == "ppo":
            lo, hi = PPO_CLIP
            return np.mean(-np.minimum(r * A, np.clip(r, lo, hi) * A))
        return np.mean(-w * A * logp_y)                            # 8: step 0: mean(-A * logp_y) = 0.490

    def backward(self, verbose=False):
        acts = self.acts
        b, p, y, A, w = acts["b"], acts["p"], acts["y"], acts["A"], acts["w"]

        # 8 -> 4: loss -> logits. d(-log p_y)/d logits = p - y_onehot; w and A are frozen numbers,
        # so they only scale each row. THIS LINE IS THE ONLY PLACE PPO AND CISPO DIFFER.
        y_onehot = np.zeros_like(p)
        y_onehot[np.arange(b), y] = 1                  # rows wrote A, B, C, C
        # p - y_onehot                                 = [[-0.856,  0.223,  0.633],
        #                                                 [ 0.144, -0.777,  0.633],
        #                                                 [ 0.144,  0.223, -0.367],
        #                                                 [ 0.144,  0.223, -0.367]]
        # A                                            = [1.732, -0.577, -0.577, -0.577]
        g_logits = (w * A)[:, None] * (p - y_onehot) / b
                                                       # step 0 (w = 1):
                                                       # [[-0.371,  0.097,  0.274],   row 1, A > 0: logit of A pulled up
                                                       #  [-0.021,  0.112, -0.091],   rows 2-4, A < 0: the written token pushed down
                                                       #  [-0.021, -0.032,  0.053],
                                                       #  [-0.021, -0.032,  0.053]]

        # 3: logits = H @ W2  ->  dW2 = H.T @ g_logits, and send the gradient back to H.
        dW2 = acts["H"].T @ g_logits                   # [[ 0.212, -0.071, -0.141],
                                                       #  [-0.150,  0.050,  0.100],
                                                       #  [-0.374,  0.125,  0.249],
                                                       #  [-0.320,  0.107,  0.213]]
        g_H = g_logits @ self.W2.T

        # 2: H = tanh(Z1). Slope of tanh is 1 - tanh^2 = [0.760, 0.880, 0.256, 0.455] in every row.
        g_Z1 = g_H * (1 - acts["H"] ** 2)

        # 1: Z1 = X @ W1  ->  dW1 = X.T @ g_Z1
        dW1 = acts["X"].T @ g_Z1                       # [[ 0.000,  0.000, 0.000, 0.000],   input 1 is 0: no gradient
                                                       #  [-0.044, -0.367, 0.071, 0.226]]

        if verbose:
            print("r        :", acts["r"])
            print("w        :", w)
            print("w * A    :", w * A)
            print("g_logits :\n", g_logits)
        return {"dW1": dW1, "dW2": dW2}

## The rollout

The rollout weights write 4 responses. Token A is the right answer and the model gives it
p = 0.144, so it is the rare token that wins (like "Wait" or "Recheck" in the MiniMax-M1
paper). We fix the 4 samples to `[A, B, C, C]` so the group has one winner.

In [2]:
net0 = Network()
p_old = net0.probs(X)
y = np.array([0, 1, 2, 2])                             # written tokens: A, B, C, C
logp_old = np.log(p_old[np.arange(4), y])              # stored by the rollout, never updated
R = np.array([1.0, 0.0, 0.0, 0.0])                     # only A is correct
A = (R - R.mean()) / R.std()                           # GRPO advantage, one per response
print("p_old     :", p_old[0])
print("written   :", [TOKENS[i] for i in y])
print("logp_old  :", logp_old)
print("reward    :", R, "  A:", A)

p_old     : [0.144 0.223 0.633]
written   : ['A', 'B', 'C', 'C']
logp_old  : [-1.937 -1.501 -0.457 -0.457]
reward    : [1. 0. 0. 0.]   A: [ 1.732 -0.577 -0.577 -0.577]


## Step 0: on-policy, all three agree

The current weights are the rollout weights, so `r = 1` in every row. PPO does not clip and
CISPO's weight is 1: all three losses give the same gradient.

In [3]:
grads = {}
for kind in ["reinforce", "ppo", "cispo"]:
    net = Network()
    net.forward(X, y, logp_old, A, kind)
    grads[kind] = net.backward(verbose=(kind == "cispo"))
for kind in ["ppo", "cispo"]:
    same = all(np.allclose(grads[kind][k], grads["reinforce"][k]) for k in ["dW1", "dW2"])
    print(f"{kind} == reinforce at step 0: {same}")

r        : [1. 1. 1. 1.]
w        : [1. 1. 1. 1.]
w * A    : [ 1.732 -0.577 -0.577 -0.577]
g_logits :
 [[-0.371  0.097  0.274]
 [-0.021  0.112 -0.091]
 [-0.021 -0.032  0.053]
 [-0.021 -0.032  0.053]]
ppo == reinforce at step 0: True
cispo == reinforce at step 0: True


## Keep stepping on the same rollout

MiniMax-M1 took 16 optimizer steps per rollout. `logp_old` stays fixed, the weights move, and
`r` drifts away from 1. Each step: forward, backward, `W = W - lr * dW`.

In [4]:
def train(kind, steps=16, lr=0.3):
    net = Network()
    rows = []
    for step in range(steps):
        net.forward(X, y, logp_old, A, kind)
        rows.append((step, net.acts["r"][0], net.acts["w"][0], net.acts["w"][1:], net.acts["p"][0, 0]))
        g = net.backward()
        net.W1 -= lr * g["dW1"]
        net.W2 -= lr * g["dW2"]
    return net, rows

for kind in ["ppo", "cispo"]:
    net, rows = train(kind)
    print(f"{kind.upper()}   step  r(row A)  w(row A)  w(rows B, C, C)        p(A)")
    for step, rA, wA, w_rest, pA in rows:
        if step in (0, 1, 2, 4, 8, 15):
            print(f"        {step:4d}  {rA:8.3f}  {wA:8.3f}  {w_rest}  {pA:.3f}")
    print(f"        after 16 steps: p = {net.probs(X)[0]}\n")

PPO   step  r(row A)  w(row A)  w(rows B, C, C)        p(A)
           0     1.000     1.000  [1. 1. 1.]  0.144
           1     1.456     0.000  [0.851 0.949 0.949]  0.210
           2     1.558     0.000  [0.    0.943 0.943]  0.225
           4     1.757     0.000  [0.808 0.895 0.895]  0.253
           8     2.263     0.000  [0. 0. 0.]  0.326
          15     2.263     0.000  [0. 0. 0.]  0.326
        after 16 steps: p = [0.326 0.177 0.497]

CISPO   step  r(row A)  w(row A)  w(rows B, C, C)        p(A)
           0     1.000     1.000  [1. 1. 1.]  0.144
           1     1.456     1.456  [0.851 0.949 0.949]  0.210
           2     2.267     2.267  [0.694 0.819 0.819]  0.327
           4     5.080     4.000  [0.305 0.315 0.315]  0.733
           8     6.435     4.000  [0.1   0.079 0.079]  0.928
          15     6.728     4.000  [0.046 0.031 0.031]  0.970
        after 16 steps: p = [0.972 0.009 0.018]



**PPO.** At step 1, row A's ratio is 1.456 > 1.28 with A > 0, so its weight becomes 0. The token
that won gets no more direct gradient, which is the "clipped out after the first on-policy update"
the paper describes. p(A) still rises slowly, because the losing rows push B and C down. From
step 8 every row is outside the band, all weights are 0, and training on this rollout stops at
p(A) = 0.326.

**CISPO.** Row A keeps its weight, `w = r`, until r passes 4 at step 4. From then on the weight
stays at 4: still trained, but capped. With no lower bound, the losing rows' weights shrink with
their ratios (r -> 0.03), so their push fades by itself. p(A) reaches 0.972.

The PPO clip is a trust region: it stops the policy from moving far from the rollout policy.
CISPO drops that limit and relies only on the cap of 4.

## Check against PyTorch

The gradient check runs at states where the weights differ: PPO after 1 step (row A clipped) and
CISPO after 4 steps (row A capped at 4). PyTorch gets the losses as written, with
`.detach()` for CISPO's stop-gradient and `torch.minimum` for PPO.

In [5]:
def torch_grads(net, kind):
    W1 = torch.tensor(net.W1, requires_grad=True)
    W2 = torch.tensor(net.W2, requires_grad=True)
    logp = torch.log_softmax(torch.tanh(torch.tensor(X) @ W1) @ W2, dim=1)
    logp_y = logp[torch.arange(4), torch.tensor(y)]
    r = torch.exp(logp_y - torch.tensor(logp_old))
    At = torch.tensor(A)
    if kind == "ppo":
        loss = (-torch.minimum(r * At, torch.clamp(r, *PPO_CLIP) * At)).mean()
    else:
        w = torch.clamp(r, *CISPO_CLIP).detach()          # stop-gradient: miles tinker_losses.py:211
        loss = (-(w * logp_y * At)).mean()               # miles tinker_losses.py:212 (sum there, mean here)
    loss.backward()
    return W1.grad.numpy(), W2.grad.numpy()

for kind, steps in [("ppo", 1), ("cispo", 4)]:
    net, _ = train(kind, steps=steps)
    net.forward(X, y, logp_old, A, kind)
    g = net.backward()
    t1, t2 = torch_grads(net, kind)
    print(f"{kind} after {steps} steps: r = {net.acts['r']}  w = {net.acts['w']}")
    print(f"   dW1 matches: {np.allclose(g['dW1'], t1)}   dW2 matches: {np.allclose(g['dW2'], t2)}")

ppo after 1 steps: r = [1.456 0.851 0.949 0.949]  w = [0.    0.851 0.949 0.949]
   dW1 matches: True   dW2 matches: True
cispo after 4 steps: r = [5.08  0.305 0.315 0.315]  w = [4.    0.305 0.315 0.315]
   dW1 matches: True   dW2 matches: True


## Why the stop-gradient matters

Without `.detach()`, autograd would also differentiate `w = r = p_now / p_old`, which contains
the current weights. The gradient per row becomes `(r + r * log p_y) * A * ∇log p_y` (product rule)
instead of `w * A * ∇log p_y`. That extra `r * log p_y` term has no meaning in the objective,
which is why CISPO freezes `w`.

In [6]:
net, _ = train("cispo", steps=1)                       # r = [1.456, 0.851, 0.949, 0.949], below the cap
W1 = torch.tensor(net.W1, requires_grad=True)
W2 = torch.tensor(net.W2, requires_grad=True)
logp_y = torch.log_softmax(torch.tanh(torch.tensor(X) @ W1) @ W2, dim=1)[torch.arange(4), torch.tensor(y)]
r = torch.exp(logp_y - torch.tensor(logp_old))
(-(r * logp_y * torch.tensor(A))).mean().backward()    # no .detach()
net.forward(X, y, logp_old, A, "cispo")
print("dW2 without detach matches CISPO:", np.allclose(net.backward()["dW2"], W2.grad.numpy()))

dW2 without detach matches CISPO: False


## Where this is in miles, and what it means for our runs

| Step above | miles CISPO (`miles/backends/training_utils/loss/hub/tinker_losses.py`) |
|---|---|
| 5. `logp_y` | `log_probs = _target_logprobs(...)`, line 200 |
| 6. `r` | `ratio = torch.exp(log_prob - sampling_log_prob)`, line 210 |
| 7. `w` | `coefficient = torch.clamp(ratio, clip_low, clip_high).detach()`, line 211 |
| 8. loss | `-(coefficient * log_prob * advantage * mask).sum()`, line 212 |

Line numbers are from branch `deniz/titan-glm53-vl-on-main-lossfix` at 0b28d4fd71. miles reads
`logp_old` from `rollout_log_probs`, the **inference engine's** log-probs.

Our runs take one optimizer step per rollout, so only step 0 above happens. With
`logp_old` from the trainer, `r = 1` and PPO, CISPO and REINFORCE give the same gradient. With
`logp_old` from the engine, `r` only reflects the trainer-vs-engine numeric gap, close to 1,
and CISPO becomes a capped weight on that gap: the same shape as TIS (`--use-tis`, cap 2.0 by
default), which our GLM runs already use.